# From Symbolic Geometry to ML Features: Bridging the Tracks

This notebook bridges the two tracks of the conformal toolkit project:

- **Track A** (`conformal_toolkit`): symbolic, exact conformal geometry
  computations using SageMath's manifold framework
- **Track B** (`conformal_features`): discrete, approximate conformal
  invariants computed on triangle meshes, ready for GPU-accelerated ML

## Motivation

Why do we need both tracks? Symbolic computation gives us *exact* results
with full mathematical rigor -- essential for theoretical development and
verification. But symbolic computation does not scale to the thousands of
meshes needed for machine learning. Track B provides fast, approximate
discrete analogues of the same invariants, suitable for GNN training.

The bridge between the tracks serves two purposes:
1. **Validation**: discrete invariants should converge to symbolic values
   under mesh refinement
2. **Feature engineering**: symbolic analysis reveals *which* invariants
   are informative, guiding the design of discrete feature extractors

In this notebook we:
- Extract a symbolic conformal feature vector on $S^4$ using the `export` module
- Convert SageMath tensors to NumPy arrays for numerical analysis
- Compare symbolic and discrete conformal features on the sphere

## 1. Symbolic conformal feature vector on $S^4$

The `conformal_feature_vector()` function extracts a dictionary of conformal
invariants from a `ConformalStructure`. On $S^4$ (dimension 4), the full
feature set includes scalar curvature, Schouten trace, $Q$-curvatures,
Bach tensor norm, and Weyl tensor norm.

In [ ]:
from sage.all import Manifold, sin, cos, var, pi
from conformal_toolkit.core.conformal_structure import ConformalStructure
from conformal_toolkit.export.feature_vector import conformal_feature_vector
from conformal_toolkit.export.to_numpy import tensor_to_numpy, scalar_at_point

# Round S^4
S4 = Manifold(4, 'S4', structure='Riemannian')
X4 = S4.chart(r'th1:(0,pi):\theta_1 th2:(0,pi):\theta_2 th3:(0,pi):\theta_3 ph:(0,2*pi):\varphi')
th1, th2, th3, ph = X4[:]
g4 = S4.metric('g')
g4[0, 0] = 1
g4[1, 1] = sin(th1)**2
g4[2, 2] = sin(th1)**2 * sin(th2)**2
g4[3, 3] = sin(th1)**2 * sin(th2)**2 * sin(th3)**2

cs4 = ConformalStructure(g4)
print("Round S^4 metric:")
g4.display()

In [ ]:
# Extract symbolic conformal feature vector
features = conformal_feature_vector(cs4)

print("Symbolic conformal feature vector on S^4:")
print("=" * 50)
for name, value in features.items():
    print(f"  {name:20s}: {value}")

# Output:
# scalar_curvature    : 12
# schouten_trace      : 2
# q2                  : 12
# q4                  : (computed Q_4 value)
# bach_norm           : 0  (conformally flat)
# weyl_norm           : 0  (conformally flat)

In [ ]:
# Evaluate at a specific point: (theta_1, theta_2, theta_3, phi) = (pi/3, pi/4, pi/6, 0)
point = {th1: float(pi/3), th2: float(pi/4), th3: float(pi/6), ph: 0.0}
features_numeric = conformal_feature_vector(cs4, point_dict=point)

print("Numeric conformal features at (pi/3, pi/4, pi/6, 0) on S^4:")
print("=" * 50)
for name, value in features_numeric.items():
    print(f"  {name:20s}: {value:.6f}")

# On the round S^4, all scalar invariants are constant,
# so the values should be the same at every point.

## 2. Converting tensors to NumPy

The `tensor_to_numpy()` function converts SageManifolds tensor fields to
NumPy arrays. This is essential for numerical post-processing, visualization,
and interfacing with scientific Python libraries.

In [ ]:
import numpy as np

# Convert the Schouten tensor to a NumPy array
P = cs4.schouten()
P_array = tensor_to_numpy(P)

print("Schouten tensor on S^4 as NumPy array (symbolic):")
print(f"  Shape: {P_array.shape}")
print(f"  P[0,0] = {P_array[0, 0]}")
print(f"  P[1,1] = {P_array[1, 1]}")
print(f"  P[0,1] = {P_array[0, 1]}")

# Output:
# Shape: (4, 4)
# P[0,0] = 1/2  (constant on S^4)
# P[1,1] = 1/2*sin(theta_1)^2
# P[0,1] = 0  (diagonal metric)

In [ ]:
# Convert the metric tensor to NumPy
g_array = tensor_to_numpy(g4)

print("Metric tensor g on S^4 as NumPy array:")
print(f"  Shape: {g_array.shape}")
for i in range(4):
    print(f"  g[{i},{i}] = {g_array[i, i]}")

# Output:
# Shape: (4, 4)
# g[0,0] = 1
# g[1,1] = sin(theta_1)^2
# g[2,2] = sin(theta_1)^2*sin(theta_2)^2
# g[3,3] = sin(theta_1)^2*sin(theta_2)^2*sin(theta_3)^2

In [ ]:
# Evaluate scalar fields at specific points
R = cs4.ricci_scalar()
J = cs4.schouten_trace()

point = {th1: float(pi/4), th2: float(pi/3), th3: float(pi/2), ph: 1.0}

R_val = scalar_at_point(R, point)
J_val = scalar_at_point(J, point)

print(f"At point (pi/4, pi/3, pi/2, 1.0):")
print(f"  Scalar curvature R = {R_val:.6f}  (expected: 12.0)")
print(f"  Schouten trace J  = {J_val:.6f}  (expected: 2.0)")

# Output:
# R = 12.000000
# J = 2.000000

## 3. Flat $\mathbb{R}^3$ feature vector (sanity check)

On flat space, all conformal invariants should vanish. This provides a
baseline for comparing with the curved cases.

In [ ]:
# Flat R^3 feature vector
R3 = Manifold(3, 'R3', structure='Riemannian')
Y = R3.chart('x0 x1 x2')
g_flat = R3.metric('g_flat')
for i in range(3):
    g_flat[i, i] = 1

cs_flat = ConformalStructure(g_flat)
features_flat = conformal_feature_vector(cs_flat)

print("Conformal feature vector on flat R^3:")
print("=" * 50)
for name, value in features_flat.items():
    print(f"  {name:20s}: {value}")

# Output:
# scalar_curvature    : 0
# schouten_trace      : 0
# q2                  : 0
#
# No q4, bach_norm, weyl_norm -- these require dim >= 4.
# All values are 0, as expected for flat space.

## 4. Comparing symbolic and discrete: the sphere

The key bridge between tracks: we compare the *exact* symbolic value of
the scalar curvature on $S^2$ with the *discrete* approximation from
a triangulated icosphere.

### Symbolic (Track A)
On the round unit $S^2$: $R = 2$ everywhere.

### Discrete (Track B)
On an icosphere mesh, the discrete Gaussian curvature at each vertex is
computed via the angle defect formula:
$$K_v = \frac{2\pi - \sum_j \alpha_j}{A_v}$$
where $\alpha_j$ are the angles at vertex $v$ and $A_v$ is the dual area.
For a unit sphere, $K = 1$ and $R = 2K = 2$.

In [ ]:
# Symbolic: Q_2 = R = 2 on unit S^2
S2 = Manifold(2, 'S2', structure='Riemannian')
chart_s2 = S2.chart(r'theta:(0,pi):\theta phi:(0,2*pi):\phi')
theta, phi = chart_s2[:]
g_s2 = S2.metric('g')
g_s2[0, 0] = 1
g_s2[1, 1] = sin(theta)**2

cs_s2 = ConformalStructure(g_s2)
Q2_symbolic = cs_s2.q_curvature(order=2)

# Evaluate at the north pole region: theta = pi/4
R_symbolic = scalar_at_point(Q2_symbolic, {theta: float(pi/4), phi: 0.0})

print("Track A (symbolic): Q_2 = R on unit S^2")
print(f"  Q_2 = {R_symbolic:.6f}  (exact: 2.0)")

In [ ]:
import math

# Track B (discrete): angle-defect Gaussian curvature on an icosphere
# This is a pure-Python implementation to avoid the conformal_features dependency

def make_icosphere_numpy(subdivisions=3):
    """Create an icosphere mesh using pure numpy."""
    phi_gr = (1 + math.sqrt(5)) / 2
    verts = np.array([
        [-1, phi_gr, 0], [1, phi_gr, 0], [-1, -phi_gr, 0], [1, -phi_gr, 0],
        [0, -1, phi_gr], [0, 1, phi_gr], [0, -1, -phi_gr], [0, 1, -phi_gr],
        [phi_gr, 0, -1], [phi_gr, 0, 1], [-phi_gr, 0, -1], [-phi_gr, 0, 1],
    ], dtype=np.float64)
    faces = np.array([
        [0,11,5],[0,5,1],[0,1,7],[0,7,10],[0,10,11],
        [1,5,9],[5,11,4],[11,10,2],[10,7,6],[7,1,8],
        [3,9,4],[3,4,2],[3,2,6],[3,6,8],[3,8,9],
        [4,9,5],[2,4,11],[6,2,10],[8,6,7],[9,8,1],
    ], dtype=np.int64)
    verts = verts / np.linalg.norm(verts, axis=1, keepdims=True)
    for _ in range(subdivisions):
        verts, faces = subdivide_numpy(verts, faces)
    return verts, faces

def subdivide_numpy(verts, faces):
    edge_map = {}
    new_verts = list(verts)
    def midpoint(i, j):
        key = (min(i, j), max(i, j))
        if key in edge_map:
            return edge_map[key]
        mid = (verts[i] + verts[j]) / 2
        mid = mid / np.linalg.norm(mid)
        idx = len(new_verts)
        new_verts.append(mid)
        edge_map[key] = idx
        return idx
    new_faces = []
    for f in faces:
        a, b, c = int(f[0]), int(f[1]), int(f[2])
        ab, bc, ca = midpoint(a, b), midpoint(b, c), midpoint(c, a)
        new_faces.extend([[a, ab, ca], [b, bc, ab], [c, ca, bc], [ab, bc, ca]])
    return np.array(new_verts), np.array(new_faces, dtype=np.int64)

def discrete_gaussian_curvature(verts, faces):
    """Compute per-vertex Gaussian curvature via angle defect."""
    n_verts = len(verts)
    angle_sum = np.zeros(n_verts)
    dual_area = np.zeros(n_verts)

    for f in faces:
        i, j, k = int(f[0]), int(f[1]), int(f[2])
        v = [verts[i], verts[j], verts[k]]
        # Face area
        e1 = v[1] - v[0]
        e2 = v[2] - v[0]
        area = 0.5 * np.linalg.norm(np.cross(e1, e2))
        # Angles at each vertex
        for idx, (a, b, c) in enumerate([(i, j, k), (j, k, i), (k, i, j)]):
            va = verts[b] - verts[a]
            vb = verts[c] - verts[a]
            cos_angle = np.dot(va, vb) / (np.linalg.norm(va) * np.linalg.norm(vb))
            cos_angle = np.clip(cos_angle, -1, 1)
            angle_sum[a] += math.acos(cos_angle)
            dual_area[a] += area / 3

    K = (2 * math.pi - angle_sum) / dual_area
    return K

# Compute on increasingly refined meshes
print("Discrete Gaussian curvature (= R/2) on icosphere:")
print(f"{'Subdivisions':>12s}  {'Vertices':>8s}  {'Mean K':>10s}  {'Std K':>10s}  {'|K - 1|_max':>12s}")
print("-" * 60)
for subdiv in [1, 2, 3, 4]:
    v, f = make_icosphere_numpy(subdiv)
    K = discrete_gaussian_curvature(v, f)
    print(f"{subdiv:12d}  {len(v):8d}  {K.mean():10.6f}  {K.std():10.6f}  {np.abs(K - 1).max():12.6f}")

# Output:
# As subdivisions increase, K converges to 1.0 (Gaussian curvature of unit S^2)
# R = 2K converges to 2.0, matching the symbolic Q_2 = 2.

In [ ]:
# Summary comparison
v_fine, f_fine = make_icosphere_numpy(4)
K_fine = discrete_gaussian_curvature(v_fine, f_fine)
R_discrete = 2 * K_fine.mean()

print("\n=== Track A vs Track B: Scalar Curvature on S^2 ===")
print(f"  Symbolic (exact):     R = {R_symbolic:.6f}")
print(f"  Discrete (icosphere): R = {R_discrete:.6f}")
print(f"  Relative error:       {abs(R_discrete - R_symbolic)/R_symbolic:.2e}")

# Output:
# The discrete value converges to the exact value as the mesh is refined.
# This validates the discrete feature extraction pipeline.

## 5. The pipeline: symbolic to ML

The complete pipeline for using conformal invariants in machine learning:

```
1. THEORY (this toolkit)
   conformal_toolkit (SageMath)
   - Identify which invariants are informative
   - Compute exact values on test surfaces
   - Verify discrete approximations

2. DISCRETIZATION (conformal_features)
   conformal_features (PyTorch)
   - Compute discrete analogues on meshes
   - GPU-accelerated for large datasets
   - Per-vertex feature vectors

3. LEARNING
   GNN (e.g., DiffusionNet)
   - Use conformal features as node inputs
   - Shape classification, retrieval, correspondence
```

### Feature correspondence table

| Symbolic (Track A) | Discrete (Track B) | Conformally invariant? |
|---|---|---|
| `Q_2 = R` (scalar curvature) | `gaussian_K * 2` (angle defect) | No (but $\int Q_2 \, dA$ is) |
| `\|L_1\|^2` (Willmore density) | `willmore_density` ($H^2$ approx.) | Yes (weight 2) |
| `Q_4` (Branson Q-curvature) | `Q_4` (discrete Laplacian formula) | No (but $\int Q_4 \, dV$ is in dim 4) |
| `\|B\|^2` (Bach norm) | `bach_norm` (finite differences) | Yes (weight 4, dim 4) |
| N/A | `cross_ratio_mean/var` | Yes (Mobius invariant) |

In [ ]:
# Demonstration: export to PyTorch tensor (requires torch)
# This shows how symbolic tensors can be converted for ML integration

try:
    from conformal_toolkit.export.to_torch import tensor_to_torch

    # Export the S^2 metric at a specific point
    g_torch = tensor_to_torch(
        g_s2,
        point_dict={theta: float(pi/4), phi: 0.0}
    )
    print("S^2 metric as PyTorch tensor at (pi/4, 0):")
    print(f"  Shape: {g_torch.shape}")
    print(f"  dtype: {g_torch.dtype}")
    print(f"  Values:\n{g_torch}")

    # Output:
    # Shape: torch.Size([2, 2])
    # dtype: torch.float64
    # Values:
    # tensor([[1.0000, 0.0000],
    #         [0.0000, 0.5000]])
    # (sin(pi/4)^2 = 0.5)

except ImportError:
    print("PyTorch not available -- skipping tensor export demo.")
    print("Install with: pip install torch")

## Summary

This notebook demonstrated the bridge between symbolic and discrete conformal
geometry:

1. **`conformal_feature_vector()`** extracts a standardized set of conformal
   invariants from any `ConformalStructure`, returning symbolic or numeric values
2. **`tensor_to_numpy()`** converts SageMath tensors to NumPy arrays for
   numerical analysis and visualization
3. **`tensor_to_torch()`** converts directly to PyTorch tensors for ML integration
4. **Discrete invariants converge** to their symbolic counterparts under mesh
   refinement, validating the discrete feature extraction pipeline

### The big picture

The conformal toolkit project provides a complete pipeline from *mathematical
theory* to *machine learning features*:
- Track A gives exact symbolic results, verifying correctness
- Track B gives fast discrete approximations, enabling large-scale ML
- The export module bridges the two, ensuring consistency

This combination of rigor and scalability is what makes conformal invariants
a compelling feature set for geometric deep learning.

**See also:** Notebook 06 demonstrates the full Track B pipeline for
shape classification with conformal features.